In [ ]:

from pathlib import Path
import pandas as pd

# Find our main project folder
project_path = Path.cwd().parent

# Locate the BPI dataset folder
bpi_path = project_path / "data : bpi2019"

# Show the files inside
print("BPI folder:", bpi_path)

for file in bpi_path.iterdir():
    print(file.name)


In [ ]:
bpi_file = bpi_path / "BPI_Challenge_2019.csv"


In [ ]:

from pathlib import Path

# Show the BPI folder location
print("BPI folder:", bpi_path)

# Check whether this folder exists
print("Folder exists:", bpi_path.exists())

# List everything inside the folder
if bpi_path.exists():
    for file in bpi_path.iterdir():
        print("Found:", repr(file.name))
else:
    print("BPI folder was not found.")


In [ ]:

bpi_df = pd.read_csv(
    bpi_file,
    encoding="cp1252",
    low_memory=False
)

print("Dataset shape:", bpi_df.shape)
display(bpi_df.head())


In [ ]:

# Set the correct CSV file path
bpi_file = bpi_path / "data : bpi2019.csv"

# Confirm that Python can find the file
print("File exists:", bpi_file.exists())

# Load the procurement event log
bpi_df = pd.read_csv(bpi_file, low_memory=False)

# Display the first five records
display(bpi_df.head())

# Check total rows and columns
print("Dataset shape:", bpi_df.shape)

# Inspect column names and data types
bpi_df.info()


In [ ]:

# Read a small sample of the file as raw bytes
with open(bpi_file, "rb") as file:
    sample = file.read(300)

print("First 300 bytes:")
print(sample)

print("\nFile size in MB:")
print(round(bpi_file.stat().st_size / (1024 * 1024), 2))


In [ ]:

# Read a small sample with Windows-1252 encoding
bpi_sample = pd.read_csv(
    bpi_file,
    encoding="cp1252",
    nrows=1000,
    low_memory=False,
    skipinitialspace=True
)

print("Sample shape:", bpi_sample.shape)

display(bpi_sample.head())

print("\nColumn names:")
for column in bpi_sample.columns:
    print(repr(column))


In [ ]:

print("Number of columns:", len(bpi_sample.columns))
print("Number of sample rows:", len(bpi_sample))

print("\nFirst 10 columns:")
print(bpi_sample.columns[:10].tolist())

print("\nLast 10 columns:")
print(bpi_sample.columns[-10:].tolist())


In [ ]:

# Remove spaces at the beginning and end of column names
bpi_sample.columns = bpi_sample.columns.str.strip()

# Verify important columns
print(bpi_sample.columns.tolist())


In [ ]:

# Count procurement activities in the sample
activity_counts = (
    bpi_sample["event concept:name"]
    .value_counts(dropna=False)
)

display(activity_counts)

print(
    "Unique activities in sample:",
    bpi_sample["event concept:name"].nunique()
)


In [ ]:

# Count distinct procurement cases in the sample
total_cases = bpi_sample["case concept:name"].nunique()

# Count events in the sample
total_events = len(bpi_sample)

# Calculate average events per observed case
avg_events = total_events / total_cases

print("Events in sample:", total_events)
print("Unique cases in sample:", total_cases)
print("Average observed events per case:", round(avg_events, 2))

# Events recorded for each case
events_per_case = (
    bpi_sample.groupby("case concept:name")
    .size()
    .sort_values(ascending=False)
)

display(events_per_case.head(10))


In [ ]:

# Convert timestamp strings to datetime
bpi_sample["event_timestamp"] = pd.to_datetime(
    bpi_sample["event time:timestamp"],
    errors="coerce",
    utc=True
)

print("Earliest timestamp:", bpi_sample["event_timestamp"].min())
print("Latest timestamp:", bpi_sample["event_timestamp"].max())

print(
    "Invalid or missing timestamps:",
    bpi_sample["event_timestamp"].isna().sum()
)

display(
    bpi_sample[
        [
            "case concept:name",
            "event concept:name",
            "event_timestamp"
        ]
    ].head(10)
)


In [ ]:

# Inspect original timestamp strings
print("First 20 original timestamps:")

for value in bpi_sample["event time:timestamp"].head(20):
    print(repr(value))

print("\nTimestamp data type:")
print(bpi_sample["event time:timestamp"].dtype)

print("\nAdditional non-null examples:")
print(
    bpi_sample["event time:timestamp"]
    .dropna()
    .sample(
        n=min(10, bpi_sample["event time:timestamp"].notna().sum()),
        random_state=42
    )
    .tolist()
)


In [ ]:

raw_timestamp = bpi_sample["event time:timestamp"]

print("Total sample records:", len(raw_timestamp))
print("Missing original timestamps:", raw_timestamp.isna().sum())

print("\nNumber of unique timestamp strings:")
print(raw_timestamp.nunique())

print("\nMost common original timestamps:")
print(raw_timestamp.value_counts().head(10))


In [ ]:

import csv

with open(
    bpi_file,
    "r",
    encoding="cp1252",
    newline=""
) as file:

    reader = csv.reader(file, skipinitialspace=True)

    headers = [
        column.strip()
        for column in next(reader)
    ]

    timestamp_index = headers.index(
        "event time:timestamp"
    )

    print("Original timestamp values:")

    for i, row in enumerate(reader):
        if i >= 10:
            break

        print(repr(row[timestamp_index]))


In [ ]:

from pathlib import Path

# Path to the newly added CSV file
original_file = bpi_path / "original.csv"

print("File exists:", original_file.exists())

if original_file.exists():
    print("File name:", original_file.name)
    print(
        "File size MB:",
        round(original_file.stat().st_size / 1024**2, 2)
    )


In [ ]:

from pathlib import Path

# Locate our BPI dataset folder
project_path = Path.cwd().parent
bpi_path = project_path / "data : bpi2019"

# Find XES files, including compressed XES files
xes_files = [
    file for file in bpi_path.rglob("*")
    if file.is_file()
    and (
        file.name.lower().endswith(".xes")
        or file.name.lower().endswith(".xes.gz")
    )
]

print("XES files found:", len(xes_files))

for file in xes_files:
    print("Filename:", file.name)
    print("Size MB:", round(file.stat().st_size / 1024**2, 2))
    print("Full path:", file)


In [ ]:

import gzip
import xml.etree.ElementTree as ET

xes_file = xes_files[0]

# Open either normal or compressed XES
open_file = (
    gzip.open
    if xes_file.name.lower().endswith(".gz")
    else open
)

timestamps = []

with open_file(xes_file, "rb") as f:
    for event, element in ET.iterparse(
        f, events=("end",)
    ):
        if element.tag.split("}")[-1] == "event":

            for child in element:
                if (
                    child.attrib.get("key")
                    == "time:timestamp"
                ):
                    timestamps.append(
                        child.attrib.get("value")
                    )
                    break

            element.clear()

            if len(timestamps) >= 10:
                break

print("First 10 XES timestamps:")

for timestamp in timestamps:
    print(timestamp)


In [ ]:

import gzip
import xml.etree.ElementTree as ET

xes_file = xes_files[0]

open_file = (
    gzip.open
    if xes_file.name.lower().endswith(".gz")
    else open
)

with open_file(xes_file, "rb") as f:
    for action, element in ET.iterparse(f, events=("end",)):

        if element.tag.split("}")[-1] == "trace":

            print("CASE-LEVEL ATTRIBUTES")

            for child in element:
                name = child.tag.split("}")[-1]

                if name != "event":
                    print(
                        child.attrib.get("key"),
                        "=",
                        child.attrib.get("value")
                    )

            print("\nFIRST EVENT ATTRIBUTES")

            first_event = next(
                (
                    child
                    for child in element
                    if child.tag.split("}")[-1] == "event"
                ),
                None
            )

            if first_event is not None:
                for attribute in first_event:
                    print(
                        attribute.attrib.get("key"),
                        "=",
                        attribute.attrib.get("value")
                    )

            break
